# AI Code Auditor v4 — QLoRA Fine-tuning
**Model:** DeepSeek-Coder-6.7B | **Method:** QLoRA | **Dataset:** Big-Vul v4

- Base: Big-Vul v2 (2,137 real CVE samples)
- Synthetic: +170 samples (90 CWE-190 + 80 CWE-416)
- Strategy: Gentle merge — NO capping of dominant classes
- Total training: 2,384 samples

In [ ]:
import subprocess
print(subprocess.run(['nvidia-smi'], capture_output=True, text=True).stdout)

In [ ]:
!pip install -q transformers==4.40.2 peft==0.10.0 trl==0.8.6 bitsandbytes==0.45.3 accelerate==0.29.3 datasets==2.19.1
print('Done')

In [ ]:
import os
os.environ['CUDA_VISIBLE_DEVICES'] = '0'
os.environ['PYTORCH_CUDA_ALLOC_CONF'] = 'expandable_segments:True'
os.environ['LD_LIBRARY_PATH'] = '/usr/local/cuda/lib64:' + os.environ.get('LD_LIBRARY_PATH', '')
print('Environment set')

In [ ]:
import os, json
from collections import Counter

TRAIN_PATH = None
VAL_PATH = None
for root, dirs, files in os.walk('/kaggle/input'):
    for f in files:
        full = os.path.join(root, f)
        if f == 'train.jsonl': TRAIN_PATH = full
        if f == 'val.jsonl':   VAL_PATH = full

assert TRAIN_PATH and VAL_PATH, 'Dataset not found! Add the v4 dataset in notebook settings'
print('Train:', TRAIN_PATH)
print('Val:  ', VAL_PATH)

with open(TRAIN_PATH) as f:
    train_data = [json.loads(line) for line in f]

print('Training samples:', len(train_data))
dist = Counter(r['cwe'] for r in train_data)
for cwe, count in sorted(dist.items(), key=lambda x: -x[1]):
    print(' ', cwe, count)
synth = sum(1 for r in train_data if r.get('source') == 'synthetic')
print('Synthetic samples:', synth)

In [ ]:
BASE_MODEL    = 'deepseek-ai/deepseek-coder-6.7b-base'
OUTPUT_DIR    = '/kaggle/working/lora_adapter_v4'
LORA_R        = 16
LORA_ALPHA    = 32
LORA_DROPOUT  = 0.05
TARGET_MODULES = ['q_proj', 'k_proj', 'v_proj', 'o_proj', 'gate_proj', 'up_proj', 'down_proj']
NUM_EPOCHS    = 3
BATCH_SIZE    = 2
GRAD_ACCUM    = 8
LR            = 2e-4
MAX_SEQ_LEN   = 512
SEED          = 42
# BATCH_SIZE=2 + GRAD_ACCUM=8 = same effective batch of 16, but uses less VRAM
print('Config ready')

In [ ]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

print('CUDA:', torch.cuda.is_available(), '|', torch.cuda.get_device_name(0))
print('VRAM:', round(torch.cuda.get_device_properties(0).total_memory/1e9, 1), 'GB')

tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL, trust_remote_code=True)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = 'right'

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type='nf4',
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
)
model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL,
    quantization_config=bnb_config,
    device_map={'': 0},
    trust_remote_code=True,
    torch_dtype=torch.float16,
)
model.config.use_cache = False
print('Model loaded. VRAM:', round(torch.cuda.memory_allocated()/1e9, 1), 'GB')

In [ ]:
from peft import LoraConfig, TaskType, get_peft_model, prepare_model_for_kbit_training

model = prepare_model_for_kbit_training(model)
lora_config = LoraConfig(
    r=LORA_R,
    lora_alpha=LORA_ALPHA,
    lora_dropout=LORA_DROPOUT,
    bias='none',
    task_type=TaskType.CAUSAL_LM,
    target_modules=TARGET_MODULES,
)
model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

In [ ]:
from datasets import load_dataset

train_dataset = load_dataset('json', data_files=TRAIN_PATH, split='train')
val_dataset   = load_dataset('json', data_files=VAL_PATH,   split='train')
print('Train:', len(train_dataset), '| Val:', len(val_dataset))

In [ ]:
from transformers import TrainingArguments
from trl import SFTTrainer
import json, shutil, matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
from pathlib import Path

training_args = TrainingArguments(
    output_dir=OUTPUT_DIR,
    num_train_epochs=NUM_EPOCHS,
    per_device_train_batch_size=BATCH_SIZE,
    per_device_eval_batch_size=BATCH_SIZE,
    gradient_accumulation_steps=GRAD_ACCUM,
    gradient_checkpointing=True,
    learning_rate=LR,
    lr_scheduler_type='cosine',
    warmup_ratio=0.03,
    fp16=True,
    logging_steps=10,
    evaluation_strategy='steps',
    eval_steps=100,
    save_strategy='steps',
    save_steps=100,
    save_total_limit=3,
    load_best_model_at_end=True,
    metric_for_best_model='eval_loss',
    optim='paged_adamw_32bit',
    group_by_length=True,
    report_to='none',
    seed=SEED,
)

trainer = SFTTrainer(
    model=model,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    tokenizer=tokenizer,
    args=training_args,
    dataset_text_field='text',
    max_seq_length=MAX_SEQ_LEN,
    packing=False,
)

import os
from pathlib import Path

steps_per_epoch = len(train_dataset) // (BATCH_SIZE * GRAD_ACCUM)
print('Train samples  :', len(train_dataset))
print('Steps per epoch:', steps_per_epoch)
print('Total steps    :', steps_per_epoch * NUM_EPOCHS)

# Auto-resume from latest checkpoint if one exists
checkpoints = sorted(Path(OUTPUT_DIR).glob('checkpoint-*')) if Path(OUTPUT_DIR).exists() else []
resume_from = str(checkpoints[-1]) if checkpoints else None
if resume_from:
    print('Resuming from checkpoint:', resume_from)
else:
    print('Starting fresh training')

trainer.train(resume_from_checkpoint=resume_from)

In [ ]:
out = Path(OUTPUT_DIR)
trainer.model.save_pretrained(out)
tokenizer.save_pretrained(out)
print('Adapter saved:', list(out.iterdir()))

log = trainer.state.log_history
with open('/kaggle/working/training_log_v4.json', 'w') as f:
    json.dump(log, f, indent=2)

tl = [(e['step'], e['loss'])      for e in log if 'loss' in e and 'eval_loss' not in e]
el = [(e['step'], e['eval_loss']) for e in log if 'eval_loss' in e]
fig, ax = plt.subplots(figsize=(10, 4))
if tl:
    s, l = zip(*tl)
    ax.plot(s, l, label='Train', color='steelblue', alpha=0.8)
if el:
    s, l = zip(*el)
    ax.plot(s, l, label='Eval', color='coral', linestyle='--', marker='o')
ax.set_title('AI Code Auditor v4 — QLoRA Training Loss', fontweight='bold')
ax.set_xlabel('Step')
ax.set_ylabel('Loss')
ax.legend()
ax.grid(alpha=0.3)
plt.tight_layout()
plt.savefig('/kaggle/working/training_loss_v4.png', dpi=150)
plt.close()

shutil.make_archive('/kaggle/working/lora_adapter_v4_download', 'zip', str(out))
zip_mb = round(Path('/kaggle/working/lora_adapter_v4_download.zip').stat().st_size/1e6)
print('Zip size:', zip_mb, 'MB')
print()
print('=' * 50)
print('TRAINING COMPLETE!')
print('Files saved in /kaggle/working/')
print('  - lora_adapter_v4_download.zip (' + str(zip_mb) + ' MB)')
print('  - training_log_v4.json')
print('  - training_loss_v4.png')
print('=' * 50)

In [ ]:
# ── Auto-save outputs as a Kaggle Dataset so files survive session end ──────
# When you wake up, go to kaggle.com/datasets and find 'codeauditor-v4-model'
import os, json, subprocess
from pathlib import Path

KAGGLE_USERNAME = os.environ.get('KAGGLE_USERNAME', '')
SAVE_DIR = Path('/kaggle/working/upload_bundle')
SAVE_DIR.mkdir(exist_ok=True)

# Copy all output files into the bundle
import shutil
files_to_save = [
    '/kaggle/working/lora_adapter_v4_download.zip',
    '/kaggle/working/training_log_v4.json',
    '/kaggle/working/training_loss_v4.png',
]
for fp in files_to_save:
    if Path(fp).exists():
        shutil.copy(fp, SAVE_DIR)
        print('Copied:', fp)

# Write dataset metadata
meta = {
    'title': 'codeauditor-v4-model',
    'id': f'{KAGGLE_USERNAME}/codeauditor-v4-model',
    'licenses': [{'name': 'CC0-1.0'}]
}
with open(SAVE_DIR / 'dataset-metadata.json', 'w') as f:
    json.dump(meta, f)

# Push to Kaggle Datasets using the API
print('Uploading to Kaggle Datasets...')
result = subprocess.run(
    ['kaggle', 'datasets', 'create', '-p', str(SAVE_DIR), '--dir-mode', 'zip'],
    capture_output=True, text=True
)
print(result.stdout)
if result.returncode == 0:
    print('SUCCESS! Files saved to kaggle.com/datasets/' + KAGGLE_USERNAME + '/codeauditor-v4-model')
    print('Go there when you wake up and download the zip!')
else:
    print('Upload failed:', result.stderr)
    print('Files are still in /kaggle/working/ — use Save Version instead')

In [ ]:
# ── Fallback: Save Version (always works, no API needed) ────────────────────
# This saves ALL /kaggle/working/ files as a notebook output version.
# Go to your notebook → Output tab to download files anytime.
print('Saving notebook version with all output files...')
from IPython.display import FileLink, display
display(FileLink('lora_adapter_v4_download.zip'))
display(FileLink('training_log_v4.json'))
display(FileLink('training_loss_v4.png'))
print()
print('IMPORTANT: Click "Save Version" button (top right) RIGHT NOW')
print('This saves all output files permanently to your notebook.')
print('Even if the session ends, files will be in the Output tab.')